> **Ghi chú xuất xứ — đọc trước.**
>
> Notebook này là **bản thăm dò N = 80 span** (61 TP / 19 FP, PlagDet nền 0.711) — chạy nhanh để kiểm tra pipeline so-sánh 6 model. Ở quy mô nhỏ này DeepSeek-V4-Flash nhỉnh hơn GLM-5.2 (0.767 vs 0.761), khoảng cách nằm trong nhiễu thống kê.
>
> **Kết quả CHÍNH THỨC dùng trong báo cáo là lần chạy Kaggle k8, N = 1000 span** (824 TP / 176 FP, nền 0.715): ở quy mô này **GLM-5.2 thắng rõ** (0.759, Δ +0.044, 0 lỗi). Artifact gốc: `kaggle_kernels/k8_model_compare/model_compare.result.json` — kernel `whaleeatu/k8-model-compare`, chạy 2026-08-13, runtime 8019s.


# So sánh 6 LLM (FPT) — verifier khử dương-tính-giả trên val

Chạy **song song** 6 model của FPT AI Marketplace trên cùng một mẫu span thật lấy từ tập
**validation** (nhãn TP/FP suy từ gold PAN — khách quan). Mỗi span: aligner tf-isf đã gắn cờ;
verifier của từng model quyết định **GIỮ/BỎ**. Đo:

- **fp_reduction** — tỉ lệ dương-tính-giả bị bỏ đúng (cao = tốt)
- **tp_retention** — tỉ lệ đúng được giữ (phải cao — đừng vứt phát hiện thật)
- **PlagDet trước → sau** khi bỏ span bị verifier reject (headline)
- độ trễ trung bình / lỗi

`N_SPANS` giới hạn số span (chi phí API). Full 5522 susp ≈ hàng nghìn span × 6 model = hàng chục giờ.

In [1]:
import sys, os, time, csv, statistics
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
_here = Path.cwd()
REPO = _here if (_here/'generation').exists() else _here.parent
sys.path.insert(0, str(REPO)); os.chdir(REPO)
os.environ['LLM_PROVIDER'] = 'fpt'
import pandas as pd
from scripts.alignment.align_tfisf import align_pair
from evaluation.plagdet import Span, plagdet_score
from generation.verify import verify_pair

MODELS = ["Llama-3.3-70B-Instruct", "DeepSeek-V4-Flash", "Qwen3.6-27B",
          "gpt-oss-20b", "gpt-oss-120b", "GLM-5.2"]
N_SPANS = 80                      # tăng nếu muốn mẫu lớn hơn (tốn API hơn)
VAL = r"C:/github/PAN2025/pan25-generated-plagiarism-detection-validation/02_validation/02_validation"
TH, TH3 = 0.30, 0.50
print(len(MODELS), "model | N_SPANS =", N_SPANS)

6 model | N_SPANS = 80


## Dựng mẫu span từ val (nhãn TP/FP từ gold — không tốn API)

In [2]:
gold_by, src_of = {}, {}
for r in csv.DictReader(open("outputs/validation_spans.csv", encoding="utf-8", newline="")):
    if r["feature"] == "plagiarism" and r["source_reference"]:
        gold_by.setdefault(r["suspicious_reference"], []).append((int(r["this_offset"]), int(r["this_length"])))
        src_of.setdefault(r["suspicious_reference"], r["source_reference"])

def rd(p): return open(p, encoding="utf-8", errors="ignore").read()
def overlaps(o, l, golds):
    e = o + l
    return any(not (e <= go or o >= go + gl) for go, gl in golds)

SPANS, gold_spans = [], []
for su, golds in gold_by.items():
    if len(SPANS) >= N_SPANS: break
    sp, rp = os.path.join(VAL, "susp", su), os.path.join(VAL, "src", src_of[su])
    if not (os.path.exists(sp) and os.path.exists(rp)): continue
    st, rt = rd(sp), rd(rp)
    pred = align_pair(st, rt, TH, TH, TH3, 4)
    if not pred: continue
    for go, gl in golds: gold_spans.append(Span(su, go, gl))
    for s, l, ss, sl in pred:
        if len(SPANS) >= N_SPANS: break
        SPANS.append({"doc": su, "s": s, "l": l, "susp": st[s:s+l], "src": rt[ss:ss+sl],
                      "tp": overlaps(s, l, golds)})

n_tp = sum(x["tp"] for x in SPANS); n_fp = len(SPANS) - n_tp
pred_before = [Span(x["doc"], x["s"], x["l"]) for x in SPANS]
pd_before = plagdet_score(gold_spans, pred_before)
print(f"{len(SPANS)} span ({n_tp} TP, {n_fp} FP) từ {len(set(x['doc'] for x in SPANS))} tài liệu")
print(f"PlagDet trước verifier: {pd_before.plagdet:.3f} (P={pd_before.precision:.3f} R={pd_before.recall:.3f})")

80 span (61 TP, 19 FP) từ 9 tài liệu
PlagDet trước verifier: 0.711 (P=0.617 R=0.843)


## Chạy song song 6 model — verifier

In [3]:
def run_model(model):
    keep_flags, confs, lats, fails = [], [], [], 0
    for x in SPANS:
        t = time.time()
        try:
            v = verify_pair(x["susp"], x["src"], model=model)
            keep = bool(v["is_plagiarism"]); c = v.get("confidence")
        except Exception:
            keep, c, fails = True, None, fails + 1     # lỗi -> giữ (an toàn)
        keep_flags.append(keep); lats.append(time.time()-t)
        if isinstance(c, (int, float)): confs.append(c)
    keep_tp = sum(1 for x, k in zip(SPANS, keep_flags) if x["tp"] and k)
    keep_fp = sum(1 for x, k in zip(SPANS, keep_flags) if not x["tp"] and k)
    pred_after = [Span(x["doc"], x["s"], x["l"]) for x, k in zip(SPANS, keep_flags) if k]
    a = plagdet_score(gold_spans, pred_after)
    return {"model": model,
            "fp_reduction": round((n_fp - keep_fp)/n_fp, 3) if n_fp else None,
            "tp_retention": round(keep_tp/n_tp, 3) if n_tp else None,
            "plagdet_before": round(pd_before.plagdet, 3),
            "plagdet_after": round(a.plagdet, 3),
            "delta": round(a.plagdet - pd_before.plagdet, 3),
            "prec_after": round(a.precision, 3),
            "avg_lat_s": round(statistics.mean(lats), 1),
            "avg_conf": round(statistics.mean(confs), 2) if confs else None,
            "errors": fails}

t0 = time.time()
with ThreadPoolExecutor(max_workers=len(MODELS)) as ex:
    res = list(ex.map(run_model, MODELS))
print(f"xong trong {time.time()-t0:.0f}s")
df = pd.DataFrame(res).sort_values(["plagdet_after", "fp_reduction"], ascending=False).reset_index(drop=True)
df

xong trong 931s


,model,fp_reduction,tp_retention,plagdet_before,plagdet_after,delta,prec_after,avg_lat_s,avg_conf,errors
0,DeepSeek-V4-Flash,0.579,0.951,0.711,0.767,0.055,0.711,11.6,0.95,0
1,GLM-5.2,0.632,0.918,0.711,0.761,0.050,0.727,7.8,0.96,0
2,gpt-oss-20b,0.526,0.951,0.711,0.749,0.038,0.710,2.4,0.93,0
3,Qwen3.6-27B,0.632,0.918,0.711,0.747,0.035,0.726,11.6,0.96,0
4,gpt-oss-120b,0.316,0.902,0.711,0.735,0.023,0.670,4.2,0.94,0
5,Llama-3.3-70B-Instruct,0.263,0.918,0.711,0.734,0.023,0.660,3.7,0.81,0


## Bảng so sánh (sắp theo PlagDet sau verifier)

In [4]:
cols = ["model", "plagdet_before", "plagdet_after", "delta", "fp_reduction", "tp_retention",
        "prec_after", "avg_conf", "avg_lat_s", "errors"]
table = df[cols].copy()
print(f"6 model FPT · {len(SPANS)} span val ({n_tp} TP / {n_fp} FP) · PlagDet nền {pd_before.plagdet:.3f}")
table

6 model FPT · 80 span val (61 TP / 19 FP) · PlagDet nền 0.711


,model,plagdet_before,plagdet_after,delta,fp_reduction,tp_retention,prec_after,avg_conf,avg_lat_s,errors
0,DeepSeek-V4-Flash,0.711,0.767,0.055,0.579,0.951,0.711,0.95,11.6,0
1,GLM-5.2,0.711,0.761,0.050,0.632,0.918,0.727,0.96,7.8,0
2,gpt-oss-20b,0.711,0.749,0.038,0.526,0.951,0.710,0.93,2.4,0
3,Qwen3.6-27B,0.711,0.747,0.035,0.632,0.918,0.726,0.96,11.6,0
4,gpt-oss-120b,0.711,0.735,0.023,0.316,0.902,0.670,0.94,4.2,0
5,Llama-3.3-70B-Instruct,0.711,0.734,0.023,0.263,0.918,0.660,0.81,3.7,0


### Đọc bảng
- **plagdet_after / delta**: model nào đẩy PlagDet lên cao nhất sau khi khử báo giả = tốt nhất cho vai trò #3.
- **tp_retention** phải ~1.0: model bỏ nhầm phát hiện thật (tp_retention thấp) là xấu dù fp_reduction cao.
- Nhãn TP/FP suy từ **gold PAN** (khách quan, không phải LLM tự chấm).
- Tăng `N_SPANS` để mẫu lớn hơn; full val (~nghìn span × 6 model) tốn hàng chục giờ + rate limit.